If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec17_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 17: Comparing Distributions
v.ekc

Three case studies today — jury panels in two counties and Mendel's pea plants — all answering one question: **is the data consistent with the model?** Plus a new statistic (TVD) for comparing whole distributions. (Slides: KL Lecture 17 deck.)

**Today's sections:**
1. Sample proportions
2. Swain v. Alabama, revisited
3. Mendel and pea flowers
4. Alameda County jury panels
5. Total Variation Distance

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## Sample Proportions

About 90% of people in the population are right handed. If I randomly sample 100 people, what proportion is right handed and what proportion is left handed?

In [ ]:
handedness_proportions = make_array(0.9,0.1)
sample_proportions(100,handedness_proportions)

---
## 2. Swain v. Alabama, Revisited

Same setup as last time — 26% eligible, 8 observed on a panel of 100. Run it again and watch where 8 falls.

In [ ]:
population_proportions = make_array(.26, .74)
population_proportions

In [ ]:
sample_proportions(100, population_proportions)

In [ ]:
def panel_proportion():
    return sample_proportions(100, population_proportions).item(0)

In [ ]:
panel_proportion()

In [ ]:
panels = make_array()

for i in np.arange(10000):
    new_panel = panel_proportion() * 100
    panels = np.append(panels, new_panel)

In [ ]:
Table().with_column(
    'Number of Black Men on Panel of 100', panels
).hist(bins=np.arange(5.5,40.))

# Plotting details; ignore this code
plots.ylim(-0.002, 0.09)
plots.scatter(8, 0, color='red', s=30);

---
## 3. Mendel and Pea Flowers 🌸

Mendel's genetic model predicts 75% purple-flowering plants. He grew 929 plants and observed 709 purple (76.3%). Close to 75 — but *how close is close enough?* The assessment recipe (KL deck, slide 21):

> **Steps in assessing a model:** ① choose a statistic (here: |sample % purple − 75|, so big values = bad for the model) → ② simulate the statistic under the model → ③ compare the observed statistic to the simulation.

In [ ]:
## Mendel had 929 plants, of which 709 had purple flowers
observed_purples = 709 / 929
observed_purples

In [ ]:
predicted_proportions = make_array(.75, .25)
sample_proportions(929, predicted_proportions)

In [ ]:
def purple_flowers():
    return sample_proportions(929, predicted_proportions).item(0) * 100

In [ ]:
purple_flowers()

In [ ]:
purples = make_array()

for i in np.arange(10000):
    new_purple = purple_flowers()
    purples = np.append(purples, new_purple)

In [ ]:
Table().with_column('Percent of purple flowers in sample of 929', purples).hist()

In [ ]:
# Plot statistic
Table().with_column('Discrepancy in sample of 929 if the model is true', abs(purples- 75)).hist(bins = make_array(0,1,2,3,4,5))
plots.ylim(-0.02, 0.5)
plots.scatter(abs(observed_purples * 100 - 75), 0, color='red', s=40);

> **Verdict for Mendel:** the observed discrepancy (about 1.3 percentage points) sits comfortably inside the simulated distribution — the data are **consistent** with his model. Note the contrast with Swain: same recipe, opposite conclusion.

---
## 4. Alameda County Jury Panels

A 2010 ACLU study of 1,453 panelists across 11 felony trials in Alameda County (KL deck, slides 25–27). Now the model involves **five** ethnicity categories at once — one number per category won't do.

In [ ]:
jury = Table().with_columns(
    'Ethnicity', make_array('Asian', 'Black', 'Latino', 'White', 'Other'),
    'Eligible', make_array(0.15, 0.18, 0.12, 0.54, 0.01),
    'Panels', make_array(0.26, 0.08, 0.08, 0.54, 0.04)
)

jury

In [ ]:
jury.barh('Ethnicity')

In [ ]:
# Under the model, this is the true distribution of people
# from which the jurors are randomly sampled
model = make_array(0.15, 0.18, 0.12, 0.54, 0.01)

In [ ]:
# Let's simulate a random draw of 1423 jurors from this distribution
simulated = sample_proportions(1423, model)
simulated

In [ ]:
# The actual observed distribution (Panels) looks quite different
# from the simulation -- try running this several times to confirm!
jury_with_simulated = jury.with_column('Simulated', simulated)
jury_with_simulated

In [ ]:
jury_with_simulated.barh('Ethnicity')

## Distance Between Distributions

In [ ]:
#  In the Mendel model from before, the difference between observed white/purple
# and their expected values (25%/75%) was our statistic.
#
# In this case, we need to understand how each of the 5 categories
# differ from their expected values according to the model.

diffs = jury.column('Panels') - jury.column('Eligible')
jury_with_difference = jury.with_column('Difference', diffs)
jury_with_difference

---
## 5. Total Variation Distance

**TVD** boils the difference between two whole distributions down to one number: sum the absolute differences, divide by 2. Result: the fraction of the population that would have to change categories to match.

### 📋 Board Reference

| Code | What it does |
|---|---|
| `abs(dist1 - dist2)` | category-by-category gaps |
| `sum(...) / 2` | the TVD |
| TVD = 0 | identical distributions |
| TVD = 1 | completely disjoint |

In [ ]:
def tvd(dist1, dist2):
    return sum(abs(dist1 - dist2))/2

### Try It 1: TVD by hand ✋

Suppose a bag should be 50% red, 30% blue, 20% green, but your sample came out 60% red, 25% blue, 15% green.

1. Compute the TVD with pencil first — then check yourself with the `tvd` function.

In [ ]:
# 1. tvd of (0.5, 0.3, 0.2) vs (0.6, 0.25, 0.15)


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Gaps: 0.1 + 0.05 + 0.05 = 0.2 → divide by 2 → 0.1.*

```python
# 1.
tvd(make_array(0.5, 0.3, 0.2), make_array(0.6, 0.25, 0.15))   # 0.1
```

</details>

In [ ]:
# The TVD of our observed data (Panels) from their expected values
# assuming the model is true (Eligbible)
obsvd_tvd = tvd(jury.column('Panels'), jury.column('Eligible'))
obsvd_tvd

In [ ]:
# The TVD of a model simluation from its expected values
tvd(sample_proportions(1423, model), jury.column('Eligible'))

In [ ]:
def simulated_tvd():
    return tvd(sample_proportions(1423, model), model)

tvds = make_array()

num_simulations = 10000
for i in np.arange(num_simulations):
    new_tvd = simulated_tvd()
    tvds = np.append(tvds, new_tvd)

In [ ]:
title = 'Simulated TVDs (if model is true)'
bins = np.arange(0, .05, .005)

Table().with_column(title, tvds).hist(bins = bins)
print('Observed TVD: ' + str(obsvd_tvd))

> **Read the result:** the observed TVD (~0.14) is far beyond anything the model produces (~0.02 and under). Same conclusion as Swain, now in five-category form: these panels don't look like random draws from the eligible population.

---
> **Today's takeaway:** the model-assessment recipe is universal — choose a statistic (a proportion, a discrepancy, a TVD), simulate it under the model, and see where the data falls. Homework 7 uses TVD exactly this way.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — TVD test
```python
def tvd(d1, d2):
    return sum(abs(d1 - d2)) / 2

def one_sim():
    return tvd(sample_proportions(n, model), model)
```